# 03. 피처 엔지니어링

이상 탐지 모델에 입력할 피처를 생성한다.

| 피처 | 설명 | 이상 의미 |
|------|------|----------|
| speed_deviation | 선박별 평균 속도 대비 편차 | 급가속/급감속 |
| course_change | 연속 레코드 간 침로 변화량 | 급격한 방향 전환 |
| signal_gap_sec | AIS 신호 간 시간 간격(초) | AIS 끄기 (은닉 행동) |
| is_night | 야간 활동 여부 (22~06시) | 야간 불법 활동 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')

from src.features import build_features

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

In [ ]:
df = pd.read_parquet('../data/ais_cleaned.parquet')
print(f'{len(df):,} rows, {df["MMSI"].nunique()} vessels')

## 3.1 피처 생성

In [ ]:
df = build_features(df)
print('생성된 피처:')
print(df[['speed_deviation', 'course_change', 'signal_gap_sec', 'is_night']].describe())

## 3.2 피처 분포 시각화

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 속도 편차
df['speed_deviation'].clip(-5, 5).hist(bins=50, ax=axes[0, 0], color='steelblue', alpha=0.7)
axes[0, 0].set_title('Speed Deviation (clipped ±5)')
axes[0, 0].axvline(0, color='red', linestyle='--')

# 침로 변화
df['course_change'].clip(upper=180).hist(bins=50, ax=axes[0, 1], color='coral', alpha=0.7)
axes[0, 1].set_title('Course Change (degrees)')
axes[0, 1].axvline(30, color='red', linestyle='--', label='30° threshold')
axes[0, 1].legend()

# 신호 간격
df['signal_gap_min'] = df['signal_gap_sec'] / 60
df['signal_gap_min'].clip(upper=120).hist(bins=50, ax=axes[1, 0], color='green', alpha=0.7)
axes[1, 0].set_title('Signal Gap (minutes, clipped at 120)')
axes[1, 0].axvline(30, color='red', linestyle='--', label='30 min threshold')
axes[1, 0].legend()

# 야간 활동 비율
night_ratio = df.groupby('MMSI')['is_night'].mean().sort_values(ascending=False)
night_ratio.hist(bins=30, ax=axes[1, 1], color='purple', alpha=0.7)
axes[1, 1].set_title('Night Activity Ratio per Vessel')
axes[1, 1].set_xlabel('Night record ratio')

plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 피처 간 상관관계

In [ ]:
feature_cols = ['SOG', 'speed_deviation', 'course_change', 'signal_gap_sec', 'is_night']
corr = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.savefig('../results/figures/feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 이상 후보 탐색 (임계값 기반)

In [ ]:
# 경험적 임계값으로 의심 행동 확인
suspicious = df[
    (df['speed_deviation'].abs() > 3) |          # 속도 편차 3σ 초과
    (df['course_change'] > 90) |                  # 90도 이상 급회전
    (df['signal_gap_sec'] > 3600)                 # 1시간 이상 신호 단절
]

print(f'전체: {len(df):,}')
print(f'의심 레코드: {len(suspicious):,} ({len(suspicious)/len(df)*100:.2f}%)')
print(f'의심 선박: {suspicious["MMSI"].nunique()}')
print()
print('의심 유형별 건수:')
print(f'  속도 이상 (|z| > 3): {(df["speed_deviation"].abs() > 3).sum():,}')
print(f'  급회전 (> 90°):      {(df["course_change"] > 90).sum():,}')
print(f'  신호 단절 (> 1hr):   {(df["signal_gap_sec"] > 3600).sum():,}')

## 3.5 피처 데이터 저장

In [ ]:
df.to_parquet('../data/ais_featured.parquet', index=False)

import os
fsize = os.path.getsize('../data/ais_featured.parquet') / 1e6
print(f'저장 완료: data/ais_featured.parquet ({fsize:.1f} MB)')